In [ ]:
# =============================================================
# NOTEBOOK 03 – REGRESSION (HỒI QUY)
# Môn: Học Máy MAT3533
# Đề tài: Hồi quy – IBM HR Analytics Employee Attrition
# Thực hiện: Phan Thanh Lâm
# Tuần: 2 | Ngày T2 – T4
# =============================================================

# ─── MỤC TIÊU ───────────────────────────────────────────────
# 1. Linear Regression baseline
# 2. Ridge Regression (L2 regularization)
# 3. Lasso Regression (L1 regularization)
# 4. MLP Regression (mở rộng)
# 5. So sánh raw data vs PCA-reduced data
# 6. 3 splits: 4:1, 7:3, 6:4

# ─── IMPORTS ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Style
# Style
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        150,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
    'axes.grid':         True,
    'grid.alpha':        0.25,
    'grid.linestyle':    '--',
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.framealpha': 0.85,
    'legend.fontsize':   9,
    'figure.facecolor':  'white',
    'axes.facecolor':    '#FAFAFA',
})

# Màu sắc
COLORS = {
    'blue':   '#534AB7',
    'red':    '#E24B4A',
    'green':  '#1D9E75',
    'orange': '#EF9F27',
    'purple': '#534AB7',
    'teal':   '#1D9E75',
    'coral':  '#E24B4A',
    'amber':  '#EF9F27',
    'pink':   '#D4537E',
}

# ─── PATHS ─────────────────────────────────────────────────
SCALED_PATH = '../data/processed/df_scaled.csv'
PCA_PATH = '../data/processed/df_pca.csv'
RESULTS_PATH = '../results'
os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs('../outputs/figures', exist_ok=True)

print("=" * 60)
print("NOTEBOOK 03 – REGRESSION (HỒI QUY)")
print("=" * 60)

# ═══════════════════════════════════════════════════════════
# BƯỚC 1 – LOAD DATA
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 1 – LOAD DATA")
print("=" * 60)

# Load scaled data (raw features)
df_scaled = pd.read_csv(SCALED_PATH)
print(f"✓ Loaded df_scaled: {df_scaled.shape}")

# Load PCA data (reduced features)
df_pca = pd.read_csv(PCA_PATH)
print(f"✓ Loaded df_pca: {df_pca.shape}")

TARGET = 'MonthlyIncome'

# Chuẩn bị data
X_raw = df_scaled.drop(columns=[TARGET, 'Attrition'], errors='ignore')
y = df_scaled[TARGET].values

# PCA data (dùng 10 components)
pca_cols = [c for c in df_pca.columns if c.startswith('PC')]
X_pca = df_pca[pca_cols].values

print(f"\n✓ Raw features: {X_raw.shape[1]} features")
print(f"✓ PCA features: {X_pca.shape[1]} components")
print(f"✓ Target: MonthlyIncome (min={y.min():.0f}, max={y.max():.0f})")

# ═══════════════════════════════════════════════════════════
# BƯỚC 2 – ĐỊNH NGHĨA MODELS & SPLITS
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 2 – ĐỊNH NGHĨA MODELS & SPLITS")
print("=" * 60)

# Các model cần train
models = {
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'Linear': LinearRegression(),
    'MLP': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42, early_stopping=True)
}

# 3 splits theo yêu cầu
splits = [
    {'name': '80:20', 'train_ratio': 0.8, 'test_ratio': 0.2},
    {'name': '70:30', 'train_ratio': 0.7, 'test_ratio': 0.3},
    {'name': '60:40', 'train_ratio': 0.6, 'test_ratio': 0.4}
]

print(f"✓ Models: {list(models.keys())}")
print(f"✓ Splits: {[s['name'] for s in splits]}")
print(f"✓ Data types: Raw, PCA")

# ═══════════════════════════════════════════════════════════
# BƯỚC 3 – TRAIN & EVALUATE
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 3 – TRAIN & EVALUATE MODELS")
print("=" * 60)

results = []

for data_name, X in [('Raw', X_raw), ('PCA', X_pca)]:
    print(f"\n📊 Training trên {data_name} data (shape={X.shape})")
    
    for split in splits:
        train_ratio = split['train_ratio']
        test_ratio = split['test_ratio']
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_ratio, random_state=42
        )
        
        print(f"\n  Split {split['name']} (train={train_ratio:.0%}, test={test_ratio:.0%})")
        
        for model_name, model in models.items():
            # Train
            model.fit(X_train, y_train)
            
            # Predict
            y_pred = model.predict(X_test)
            
            # Metrics
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            
            results.append({
                'Data': data_name,
                'Split': split['name'],
                'Model': model_name,
                'RMSE': round(rmse, 2),
                'MAE': round(mae, 2),
                'R2': round(r2, 4)
            })
            
            print(f"    {model_name:<8} | RMSE={rmse:>8.2f} | MAE={mae:>7.2f} | R2={r2:>7.4f}")

# ═══════════════════════════════════════════════════════════
# BƯỚC 4 – BẢNG KẾT QUẢ TỔNG HỢP
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 4 – BẢNG KẾT QUẢ TỔNG HỢP")
print("=" * 60)

results_df = pd.DataFrame(results)
print("\n" + results_df.to_string(index=False))

# Lưu kết quả
results_df.to_csv(f'{RESULTS_PATH}/regression_results.csv', index=False)
print(f"\n✓ Saved: {RESULTS_PATH}/regression_results.csv")

# Bảng pivot cho báo cáo
pivot_rmse = results_df.pivot_table(index=['Data', 'Split'], columns='Model', values='RMSE')
pivot_r2 = results_df.pivot_table(index=['Data', 'Split'], columns='Model', values='R2')

print("\n📊 RMSE theo Model:")
print(pivot_rmse.round(2))

print("\n📊 R² theo Model:")
print(pivot_r2.round(4))

# ═══════════════════════════════════════════════════════════
# BƯỚC 5 – VISUALIZATION
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 5 – VISUALIZATION")
print("=" * 60)

# Hình 8: So sánh RMSE giữa các models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Hình 8: So sánh RMSE giữa các Models', fontsize=13, fontweight='bold')

_MODEL_PALETTE = ['#534AB7', '#EF9F27', '#D4537E']
_MARKERS = ['o', 'D', 'v']
_LINESTYLES = ['-', '--', '-.']

# Raw data
raw_data = results_df[results_df['Data'] == 'Raw']
for idx, model in enumerate(models.keys()):
    model_data = raw_data[raw_data['Model'] == model]
    axes[0].plot(model_data['Split'], model_data['RMSE'],
                 marker=_MARKERS[idx], label=model,
                 linewidth=2.5, markersize=9,
                 color=_MODEL_PALETTE[idx],
                 linestyle=_LINESTYLES[idx],
                 markeredgecolor='white', markeredgewidth=1.2)
axes[0].set_xlabel('Split (Train:Test)')
axes[0].set_ylabel('RMSE (USD)')
axes[0].set_title('Raw Data')
axes[0].legend(loc='upper right')

# PCA data
pca_data = results_df[results_df['Data'] == 'PCA']
for idx, model in enumerate(models.keys()):
    model_data = pca_data[pca_data['Model'] == model]
    axes[1].plot(model_data['Split'], model_data['RMSE'],
                 marker=_MARKERS[idx], label=model,
                 linewidth=2.5, markersize=9,
                 color=_MODEL_PALETTE[idx],
                 linestyle=_LINESTYLES[idx],
                 markeredgecolor='white', markeredgewidth=1.2)
axes[1].set_xlabel('Split (Train:Test)')
axes[1].set_ylabel('RMSE (USD)')
axes[1].set_title('PCA Data (10 components)')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.savefig('../outputs/figures/fig08_rmse_comparison.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig08_rmse_comparison.png")

# Hình 9: Bar chart so sánh R²
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle('Hình 9: R² Score Comparison (Split 80:20)', fontsize=13, fontweight='bold')

split_80 = results_df[results_df['Split'] == '80:20']
x = np.arange(len(split_80['Model'].unique()))
width = 0.35

raw_best = split_80[split_80['Data'] == 'Raw']['R2'].values
pca_best = split_80[split_80['Data'] == 'PCA']['R2'].values

bars_raw = ax.bar(x - width/2, raw_best, width, label='Raw Data', color=COLORS['blue'], alpha=0.8)
bars_pca = ax.bar(x + width/2, pca_best, width, label='PCA Data', color=COLORS['orange'], alpha=0.8)
for bar in bars_raw:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
for bar in bars_pca:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models.keys())
ax.set_xlabel('Model')
ax.set_ylabel('R² Score')
ax.set_title('R² Score (Split 80:20)')
ax.legend()
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/figures/fig09_r2_comparison.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig09_r2_comparison.png")

# ═══════════════════════════════════════════════════════════
# BƯỚC 6 – ACTUAL VS PREDICTED (model tốt nhất)
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 6 – ACTUAL VS PREDICTED (Best Model)")
print("=" * 60)

# Tìm model tốt nhất dựa trên R²
best_row = results_df[results_df['R2'] == results_df['R2'].max()].iloc[0]
print(f"🏆 Best Model: {best_row['Model']} on {best_row['Data']} data (Split {best_row['Split']})")
print(f"   R² = {best_row['R2']:.4f}, RMSE = {best_row['RMSE']:.2f}")

# Train lại best model trên full data để visualize
if best_row['Data'] == 'Raw':
    X_best = X_raw
else:
    X_best = X_pca

X_train, X_test, y_train, y_test = train_test_split(X_best, y, test_size=0.2, random_state=42)

best_model = models[best_row['Model']]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

# Scatter plot
fig, ax = plt.subplots(figsize=(8, 8))
_errors = np.abs(y_test - y_pred)
sc = ax.scatter(y_test, y_pred, c=_errors, cmap='RdYlGn_r',
                alpha=0.65, s=35, edgecolors='none')
plt.colorbar(sc, ax=ax, label='|Error| (USD)', shrink=0.75)
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=1.8, label='Perfect Prediction')
rmse_best = np.sqrt(np.mean(_errors**2))
mae_best  = np.mean(_errors)
ax.text(0.04, 0.95,
        f'RMSE = {rmse_best:,.0f} USD\nMAE  = {mae_best:,.0f} USD\nR²   = {best_row["R2"]:.4f}',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='#cccccc', alpha=0.9))
ax.set_xlabel('Actual MonthlyIncome (USD)')
ax.set_ylabel('Predicted MonthlyIncome (USD)')
ax.set_title(f'Actual vs Predicted — {best_row["Model"]}', fontweight='bold')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('../outputs/figures/fig10_actual_vs_predicted.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig10_actual_vs_predicted.png")

# ═══════════════════════════════════════════════════════════
# TÓM TẮT CUỐI
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📊 TÓM TẮT REGRESSION")
print("=" * 60)
print(f"""
  Models trained    : {len(models)} models × 2 data types × 3 splits = {len(results)} runs
  Best model        : {best_row['Model']} on {best_row['Data']} data
  Best R²           : {best_row['R2']:.4f}
  Best RMSE         : {best_row['RMSE']:.2f} USD
  
  Output files:
    📁 results/regression_results.csv
    📁 results/tuning_results.csv
    📁 outputs/figures/fig08_rmse_comparison.png
    📁 outputs/figures/fig09_r2_comparison.png
    📁 outputs/figures/fig10_actual_vs_predicted.png
""")
print("=" * 60)
print("✅ notebook_03_regression - HOÀN THÀNH!")

NOTEBOOK 03 – REGRESSION (HỒI QUY)

BƯỚC 1 – LOAD DATA
✓ Loaded df_scaled: (1470, 48)
✓ Loaded df_pca: (1470, 12)

✓ Raw features: 46 features
✓ PCA features: 10 components
✓ Target: MonthlyIncome (min=1009, max=19999)

BƯỚC 2 – ĐỊNH NGHĨA MODELS & SPLITS
✓ Models: ['KNN', 'Linear', 'MLP']
✓ Splits: ['80:20', '70:30', '60:40']
✓ Data types: Raw, PCA

BƯỚC 3 – TRAIN & EVALUATE MODELS

📊 Training trên Raw data (shape=(1470, 46))

  Split 80:20 (train=80%, test=20%)
    KNN      | RMSE= 2118.24 | MAE=1613.63 | R2= 0.7947
    Linear   | RMSE= 1169.21 | MAE= 895.37 | R2= 0.9375
    MLP      | RMSE= 1305.36 | MAE=1032.19 | R2= 0.9220

  Split 70:30 (train=70%, test=30%)
    KNN      | RMSE= 1966.24 | MAE=1495.91 | R2= 0.8088
    Linear   | RMSE= 1146.44 | MAE= 881.75 | R2= 0.9350
    MLP      | RMSE= 1364.52 | MAE=1059.08 | R2= 0.9079

  Split 60:40 (train=60%, test=40%)
    KNN      | RMSE= 2011.22 | MAE=1518.65 | R2= 0.8083
    Linear   | RMSE= 1177.62 | MAE= 901.27 | R2= 0.9343
    MLP   